# 01 — Data Loading and Joining

This notebook loads the five raw dataset files, inspects each one, 
joins them into a single customer-level table, and saves the result 
for use in subsequent notebooks.

**Inputs:** Raw files in `../data/`  
**Output:** `../data/customers_merged.csv`

In [4]:
import pandas as pd
import json

# Load users and cards in full - they are small
users = pd.read_csv('../data/users_data.csv')
cards = pd.read_csv('../data/cards_data.csv')

# Load transactions in chunks and only keep essential columns
# This avoids loading 1.3GB into memory at once
cols_needed = ['id', 'client_id', 'card_id', 'amount', 'date', 'mcc', 'use_chip']

chunks = []
for chunk in pd.read_csv('../data/transactions_data.csv', 
                          usecols=cols_needed,
                          chunksize=500000):
    chunks.append(chunk)

transactions = pd.concat(chunks)

print("Transactions:", transactions.shape)
print("Users:", users.shape)
print("Cards:", cards.shape)
print("Memory usage:", transactions.memory_usage(deep=True).sum() / 1e6, "MB")

Transactions: (13305915, 7)
Users: (2000, 14)
Cards: (6146, 13)
Memory usage: 2936.789905 MB


In [9]:
# Aggregate transactions to customer level
# This collapses 13 million rows into 2000 rows - one per customer
customer_transactions = transactions.groupby('client_id').agg(
    total_spend=('amount', 'sum'),
    avg_transaction=('amount', 'mean'),
    transaction_count=('amount', 'count'),
    max_transaction=('amount', 'max'),
    min_transaction=('amount', 'min'),
    unique_merchants=('mcc', 'nunique')
).reset_index()

print("Customer transaction summary:", customer_transactions.shape)
print(customer_transactions.head())

del transactions
print("\nTransactions table removed from memory")

Customer transaction summary: (1219, 7)
   client_id  total_spend  avg_transaction  transaction_count  \
0          0    625799.67        48.909705              12795   
1          1    336187.37        33.375099              10073   
2          2    291534.27        27.472132              10612   
3          3    280685.46        46.773114               6001   
4          4    595722.36        39.601300              15043   

   max_transaction  min_transaction  unique_merchants  
0          1128.47           -493.0                85  
1           937.15           -491.0                76  
2           519.02           -476.0                67  
3           990.20           -384.0                62  
4          1624.15           -494.0                92  

Transactions table removed from memory


In [8]:
# Check what the amount column looks like
transactions['amount'] = transactions['amount'].str.replace('$', '', regex=False).astype(float)

print(transactions['amount'].dtype)
print(transactions['amount'].head(10))

float64
0    -77.00
1     14.57
2     80.00
3    200.00
4     46.41
5      4.81
6     77.00
7     26.46
8    261.58
9     10.74
Name: amount, dtype: float64


In [10]:
# Rename users and cards id columns to match client_id for joining
users = users.rename(columns={'id': 'client_id'})

# Join transaction summary with user profile
merged = customer_transactions.merge(users, on='client_id', how='left')

# Join cards - one customer can have multiple cards
# We take the max credit limit per customer as their financial capacity
cards_summary = cards.groupby('client_id').agg(
    total_credit_limit=('credit_limit', 'sum'),
    num_cards=('id', 'count'),
    has_dark_web_card=('card_on_dark_web', lambda x: (x == 'Yes').any().astype(int))
).reset_index()

# Final merge
merged = merged.merge(cards_summary, on='client_id', how='left')

print("Final merged table:", merged.shape)
print("Columns:", merged.columns.tolist())
print(merged.head())

Final merged table: (1219, 23)
Columns: ['client_id', 'total_spend', 'avg_transaction', 'transaction_count', 'max_transaction', 'min_transaction', 'unique_merchants', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'total_credit_limit', 'num_cards', 'has_dark_web_card']
   client_id  total_spend  avg_transaction  transaction_count  \
0          0    625799.67        48.909705              12795   
1          1    336187.37        33.375099              10073   
2          2    291534.27        27.472132              10612   
3          3    280685.46        46.773114               6001   
4          4    595722.36        39.601300              15043   

   max_transaction  min_transaction  unique_merchants  current_age  \
0          1128.47           -493.0                85           33   
1           937.15           -491.0               

In [11]:
# Clean dollar sign columns in users
for col in ['per_capita_income', 'yearly_income', 'total_debt']:
    users[col] = users[col].str.replace('$', '', regex=False).astype(float)

# Clean credit limit in cards before summarising
cards['credit_limit'] = cards['credit_limit'].str.replace('$', '', regex=False).astype(float)

# Redo cards summary now that credit_limit is numeric
cards_summary = cards.groupby('client_id').agg(
    total_credit_limit=('credit_limit', 'sum'),
    num_cards=('id', 'count'),
    has_dark_web_card=('card_on_dark_web', lambda x: (x == 'Yes').any().astype(int))
).reset_index()

# Redo the full merge with clean data
merged = customer_transactions.merge(users, on='client_id', how='left')
merged = merged.merge(cards_summary, on='client_id', how='left')

print("Columns:", merged.columns.tolist())
print(merged[['client_id', 'yearly_income', 'total_debt', 'total_credit_limit']].head())

Columns: ['client_id', 'total_spend', 'avg_transaction', 'transaction_count', 'max_transaction', 'min_transaction', 'unique_merchants', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'total_credit_limit', 'num_cards', 'has_dark_web_card']
   client_id  yearly_income  total_debt  total_credit_limit
0          0        59613.0     36199.0            105656.0
1          1        45360.0     14587.0             41805.0
2          2        27447.0     80850.0             50361.0
3          3        27943.0     18693.0             13722.0
4          4        76431.0    115362.0            136202.0


In [12]:
# Save the merged dataset
merged.to_csv('../data/customers_merged.csv', index=False)
print("Saved customers_merged.csv with shape:", merged.shape)

Saved customers_merged.csv with shape: (1219, 23)
